In [0]:
# Lab 1 / Phase 1 / Stage C2 — UC Function tool #2 (corrected: subquery instead of QUALIFY)
spark.sql("""
CREATE OR REPLACE FUNCTION workspace.default.rank_companies_by_cashflow(
    p_metric STRING COMMENT 'Which metric to rank by: one of net, inflow, outflow',
    p_top_n  INT    COMMENT 'How many company codes to return, e.g. 5'
)
RETURNS TABLE (
    rank_position      INT,
    company_code       STRING,
    currency           STRING,
    total_net_cashflow DECIMAL(38,4),
    total_inflow       DECIMAL(38,4),
    total_outflow      DECIMAL(38,4)
)
COMMENT 'Ranks SAP company codes by a chosen cash flow metric (net, inflow, or outflow) and returns the top N. Use this to answer comparison or ranking questions, such as which company codes have the highest cash flow.'
RETURN
    SELECT rank_position, company_code, currency,
           total_net_cashflow, total_inflow, total_outflow
    FROM (
        SELECT
            CAST(ROW_NUMBER() OVER (
                ORDER BY
                  CASE p_metric
                    WHEN 'inflow'  THEN SUM(total_inflow)
                    WHEN 'outflow' THEN SUM(total_outflow)
                    ELSE SUM(net_cashflow)
                  END DESC
            ) AS INT)              AS rank_position,
            company_code,
            MAX(currency)          AS currency,
            SUM(net_cashflow)      AS total_net_cashflow,
            SUM(total_inflow)      AS total_inflow,
            SUM(total_outflow)     AS total_outflow
        FROM workspace.default.gold_monthly_cashflow
        GROUP BY company_code
    )
    WHERE rank_position <= p_top_n
""")

print("Test — top 5 by net cash flow:")
spark.sql("SELECT * FROM workspace.default.rank_companies_by_cashflow('net', 5)").show()
print("Test — top 3 by inflow:")
spark.sql("SELECT * FROM workspace.default.rank_companies_by_cashflow('inflow', 3)").show()